# 1단계: MaxEnt 지도 → 지역별 전체 셀 (`all.pkl`)

① MaxEnt 서식지 적합성 지도(격자 CSV)의 각 셀에 행정구역(시군구)을 붙이고, 지역별로 나눠 저장합니다.
다음 단계 `make_sampling_pkl.ipynb`가 이 파일에서 지역마다 400셀을 뽑습니다.

### 입력
- `DATA/SDM_data/Maxent/{scenario}.csv` (12개, 열: `x, y, {scenario}`)
  — 외부에서 받은 MaxEnt 3.4.1 결과(`Trachemys_scripta_elegans_{scenario}_avg.asc`를 격자점 CSV로 변환한 것, 2025-03).
  제출 논문(Supplementary Data 2) 결과와 같은 입력입니다.
- `DATA/SDM_data/latin/latlong_ex.xlsx` — 격자 좌표 ↔ 행정구역 매핑표

### 출력
- `DATA/SDM_data/latin/TES_maxent/all.pkl` — `{시나리오: {지역: DataFrame(x, y, {시나리오}, SIG_ENG_NM)}}`
- `DATA/SDM_data/latin/local_index.csv` — 지역 목록

In [1]:
import os
import pickle
from pathlib import Path

import numpy as np
import pandas as pd

## 1. 경로 설정

In [2]:
# 이 저장소(PDSI)를 어디에 클론해도 동작하도록, 위쪽 폴더에서 "PDSI" git 저장소를 찾아
# 그 형제 폴더인 "DATA"를 데이터 루트로 씁니다. 다른 위치면 환경변수 PDSI_DATA_ROOT 로 지정하세요.
def _find_project_root(marker="PDSI"):
    p = Path.cwd().resolve()
    for candidate in [p, *p.parents]:
        if candidate.name == marker and (candidate / ".git").exists():
            return candidate
    return None

_env_override = os.environ.get("PDSI_DATA_ROOT")
if _env_override:
    base = Path(_env_override)
else:
    _root = _find_project_root()
    if _root is None:
        raise FileNotFoundError(
            "'PDSI' 저장소 폴더를 상위 경로에서 찾지 못했습니다. "
            "환경변수 PDSI_DATA_ROOT 에 DATA 폴더의 상위 경로를 직접 지정하세요."
        )
    base = _root.parent

sdm_data_path = base / 'DATA' / 'SDM_data'
maxent_path = sdm_data_path / 'Maxent'                        # ① MaxEnt 서식지 적합성 지도 (시나리오별 CSV)
latlong_file = sdm_data_path / 'latin' / 'latlong_ex.xlsx'
local_index_file = sdm_data_path / 'latin' / 'local_index.csv'
species_output_path = sdm_data_path / 'latin' / 'TES_maxent'
all_file = species_output_path / 'all.pkl'
sampling_file = species_output_path / 'sampling.pkl'

scenarios = [
    'ssp126_2030', 'ssp126_2050', 'ssp126_2070', 'ssp126_2090',
    'ssp245_2030', 'ssp245_2050', 'ssp245_2070', 'ssp245_2090',
    'ssp585_2030', 'ssp585_2050', 'ssp585_2070', 'ssp585_2090',
]
print(f'데이터 루트(base): {base}')
maxent_files = {s: maxent_path / f'{s}.csv' for s in scenarios}
missing = [s for s, f in maxent_files.items() if not f.exists()]
assert not missing, f'CSV 파일 없음: {missing}'
species_output_path.mkdir(parents=True, exist_ok=True)

데이터 루트(base): <DATA 루트>


## 2. MaxEnt 지도 읽기

12개 CSV의 격자점 `(x, y)`가 같은 순서로 같은지 확인합니다.

In [3]:
maxent = {}
ref_xy = None
for s in scenarios:
    df = pd.read_csv(maxent_files[s])
    assert list(df.columns) == ['x', 'y', s], f'{s}: 열 이름이 예상과 다릅니다 {list(df.columns)}'
    assert df[s].notna().all(), f'{s}: 결측값이 있습니다'
    if ref_xy is None:
        ref_xy = df[['x', 'y']].values
    else:
        assert np.array_equal(df[['x', 'y']].values, ref_xy), f'{s}: 격자점이 다릅니다'
    maxent[s] = df
print(f'격자 셀 {len(ref_xy)}개 — {len(scenarios)}개 시나리오 모두 동일')

격자 셀 140408개 — 12개 시나리오 모두 동일


## 3. 행정구역 매칭

In [4]:
latlong = pd.read_excel(latlong_file)
latlong = latlong.drop(columns=['OBJECTID'], errors='ignore')

# Gwangju-si, Goseong-gun, Inje-gun 동명 지역 구분
latlong.loc[(latlong['SIG_ENG_NM'] == 'Gwangju-si') & (latlong['y'] <= 36.0), 'SIG_ENG_NM'] = 'Gwangju-si1'
latlong.loc[(latlong['SIG_ENG_NM'] == 'Gwangju-si') & (latlong['y'] > 36.0), 'SIG_ENG_NM'] = 'Gwangju-si2'
latlong.loc[(latlong['SIG_ENG_NM'] == 'Goseong-gun') & (latlong['y'] <= 37.0), 'SIG_ENG_NM'] = 'Goseong-gun1'
latlong.loc[(latlong['SIG_ENG_NM'] == 'Goseong-gun') & (latlong['y'] > 37.0), 'SIG_ENG_NM'] = 'Goseong-gun2'
latlong.loc[(latlong['SIG_ENG_NM'] == 'Inje-gun') & (latlong['y'] > 37.0), 'SIG_ENG_NM'] = 'Inje-gun1'

# 좌표를 소수점 3자리로 반올림해 매칭
latlong_round = latlong.round({'x': 3, 'y': 3})

maxent_datas = {}
for s in scenarios:
    merged = pd.merge(maxent[s].round({'x': 3, 'y': 3}), latlong_round, how='inner', on=['x', 'y'])
    local_values = merged['SIG_ENG_NM'].drop_duplicates().tolist()
    maxent_datas[s] = {name: merged[merged['SIG_ENG_NM'] == name] for name in local_values}

# 모든 시나리오에서 지역별 셀 목록(좌표·순서·인덱스)이 같은지 확인
ref = maxent_datas[scenarios[0]]
for s in scenarios[1:]:
    assert list(maxent_datas[s]) == list(ref)
    for name in ref:
        assert maxent_datas[s][name].index.equals(ref[name].index)
        assert np.array_equal(maxent_datas[s][name][['x', 'y']].values, ref[name][['x', 'y']].values)

print(f'매칭된 셀: {sum(len(v) for v in ref.values())}개, 지역 수: {len(ref)}')

매칭된 셀: 140194개, 지역 수: 167


## 4. 저장

In [5]:
with open(all_file, 'wb') as f:
    pickle.dump(maxent_datas, f)
pd.Series(list(maxent_datas[scenarios[0]])).to_csv(local_index_file, index=False, header=['SIG_ENG_NM'])

print(f'✓ {all_file}')
print(f'✓ {local_index_file} ({len(maxent_datas[scenarios[0]])}개 지역)')

✓ <DATA 루트>/DATA/SDM_data/latin/TES_maxent/all.pkl
✓ <DATA 루트>/DATA/SDM_data/latin/local_index.csv (167개 지역)
